In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os
from datetime import datetime
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

In [ ]:
# Path to the folder with CSV stock files
stock_folder = '../data/raw/yfinance_data/Data/'
print(f"Looking for CSV files in: {os.path.abspath(stock_folder)}")

# Find all .csv files
stock_files = glob.glob(os.path.join(stock_folder, '*.csv'))
print(f"Found CSV files: {stock_files}")

all_stocks = []
for file in stock_files:
    ticker = os.path.basename(file).replace('.csv', '')
    if ticker.startswith('._'):   # skip Mac resource files
        continue
    print(f"Loading {ticker} from {file}")
    df = pd.read_csv(file)
    # Ensure Date column is proper
    if 'Date' in df.columns:
        df['Date'] = pd.to_datetime(df['Date'])
        df['Ticker'] = ticker
        all_stocks.append(df)
    else:
        print(f"Warning: No 'Date' column in {ticker}. Columns: {df.columns.tolist()}")

if len(all_stocks) == 0:
    raise ValueError("No stock CSV files loaded. Check folder and file names.")
    
stock_df = pd.concat(all_stocks, ignore_index=True)
print(f"\nCombined shape: {stock_df.shape}")
print(stock_df.head())
print("\nTicker distribution:")
print(stock_df['Ticker'].value_counts())

In [ ]:


news_df = pd.read_csv('../data/raw/newsData/raw_analyst_ratings.csv')
print(f"News data shape: {news_df.shape}")
print("Columns:", news_df.columns.tolist())
news_df.head()

In [ ]:
# The date column is already a datetime with timezone. Convert to naive.
# Keep the local time (Eastern) but drop the timezone offset.
news_df['date'] = news_df['date'].dt.tz_localize(None)

# Stock dates are already naive (make sure)
stock_df['Date'] = pd.to_datetime(stock_df['Date']).dt.tz_localize(None)

print(f"News date range: {news_df['date'].min()} to {news_df['date'].max()}")
print(f"Stock date range: {stock_df['Date'].min()} to {stock_df['Date'].max()}")

In [ ]:
news_df['headline_length'] = news_df['headline'].str.len()
news_df['word_count'] = news_df['headline'].str.split().str.len()

print(news_df['headline_length'].describe())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(news_df['headline_length'], bins=50, ax=axes[0])
axes[0].set_title('Headline Character Length')
sns.histplot(news_df['word_count'], bins=50, ax=axes[1])
axes[1].set_title('Headline Word Count')
plt.tight_layout()
plt.savefig('../reports/figures/headline_length_dist.png', dpi=150)
plt.show()

In [ ]:
publisher_counts = news_df['publisher'].value_counts()
print("Top 10 publishers:\n", publisher_counts.head(10))

news_df['publisher_domain'] = news_df['publisher'].str.split('@').str[-1]
domain_counts = news_df['publisher_domain'].value_counts().head(10)
print("\nTop 10 domains:\n", domain_counts)

fig, ax = plt.subplots(figsize=(12, 6))
publisher_counts.head(15).plot(kind='barh', ax=ax)
ax.set_title('Top 15 Publishers')
ax.set_xlabel('Article Count')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../reports/figures/top_publishers.png', dpi=150)
plt.show()

In [ ]:
news_df['date_only'] = news_df['date'].dt.date
daily_volume = news_df.groupby('date_only').size()

fig, ax = plt.subplots(figsize=(14, 5))
daily_volume.plot(ax=ax)
ax.set_title('Daily News Publication Volume')
ax.set_xlabel('Date')
ax.set_ylabel('Number of Articles')
plt.tight_layout()
plt.savefig('../reports/figures/daily_volume.png', dpi=150)
plt.show()

news_df['hour'] = news_df['date'].dt.hour
hourly_volume = news_df.groupby('hour').size()
fig, ax = plt.subplots(figsize=(12, 4))
hourly_volume.plot(kind='bar', ax=ax)
ax.set_title('Hourly Publication Volume (ET)')
ax.set_xlabel('Hour of Day')
plt.show()

In [ ]:
def preprocess_text(text):
    text = str(text).lower()
    # remove stock tickers like $AAPL
    text = ' '.join([word for word in text.split() if not word.startswith('$')])
    return text

news_df['clean_headline'] = news_df['headline'].apply(preprocess_text)

tfidf = TfidfVectorizer(max_features=100, stop_words='english')
tfidf_matrix = tfidf.fit_transform(news_df['clean_headline'])
feature_names = tfidf.get_feature_names_out()

avg_scores = tfidf_matrix.mean(axis=0).A1
top_indices = avg_scores.argsort()[-20:][::-1]
top_keywords = [(feature_names[i], avg_scores[i]) for i in top_indices]
print("Top 20 keywords:\n", top_keywords)

lda = LatentDirichletAllocation(n_components=8, random_state=42)
lda.fit(tfidf_matrix)

def display_topics(model, feature_names, n_top_words=8):
    topics = []
    for idx, topic in enumerate(model.components_):
        top_words = [feature_names[i] for i in topic.argsort()[-n_top_words:][::-1]]
        topics.append(f"Topic {idx+1}: {', '.join(top_words)}")
    return topics

print("\nIdentified topics:\n", display_topics(lda, feature_names))

In [ ]:
if 'stock' in news_df.columns:
    stock_counts_news = news_df['stock'].value_counts()
    print("News articles per stock:\n", stock_counts_news.head(10))
    fig, ax = plt.subplots(figsize=(10,5))
    stock_counts_news.head(10).plot(kind='bar', ax=ax)
    ax.set_title('News Volume by Stock')
    ax.set_ylabel('Article Count')
    plt.tight_layout()
    plt.savefig('../reports/figures/articles_per_stock.png', dpi=150)
    plt.show()
else:
    print("No 'stock' column in news data. Will attempt to extract from headlines later.")

In [ ]:
stock_df.to_csv('../data/processed/stock_combined.csv', index=False)
news_df.to_csv('../data/processed/news_processed.csv', index=False)
print("Processed data saved to CSV format.")